# Welcome to the OnSSET Notebook - Calibration

This Jupyter based interface is built on the [OnSSET](http://www.onsset.org/) tool developed to provide an easy and quick way to create the calibrated input file.

In [2]:
!pip install setuptools

#### Start by importing the code 

In [3]:
from onsset import *
from IPython.display import display, Markdown, HTML
%matplotlib inline
%run funcs.ipynb
import warnings
warnings.filterwarnings('ignore')

In [4]:
import matplotlib.pylab as plt
import seaborn as sns

# 1. GIS data selection

First, run the cell below to browse to the directory your input CSV file is located at and select the input file. 

In [5]:
import tkinter as tk
from tkinter import filedialog, messagebox
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
messagebox.showinfo('OnSSET', 'Open the input file with extracted GIS data')
input_file = filedialog.askopenfilename()

onsseter = SettlementProcessor(input_file)
onsseter.conditioning()

ElecPop is missing from the csv file. Filling with 0 values
Admin_1 is missing from the csv file. Filling with 0 values
Hydropower is missing from the csv file. Filling with 0 values
HydropowerFID is missing from the csv file. Filling with 9999 values
MGDist is missing from the csv file. Filling with 0 values


In [6]:
print("Actual columns in your file:")
print(onsseter.df.columns.tolist())
print("\nFirst few rows of data:")
print(onsseter.df.head())

Actual columns in your file:
['AgriDemand', 'CommercialDemand', 'Country', 'CurrentHVLineDist', 'CurrentMVLineDist', 'EducationDemand', 'GHI', 'GridCellArea', 'HealthDemand', 'HydropowerDist', 'IsUrban', 'NightLights', 'PerCapitaDemand', 'PlannedHVLineDist', 'PlannedMVLineDist', 'Pop', 'ResidentialDemandTierCustom', 'RoadDist', 'SubstationDist', 'TransformerDist', 'TravelHours', 'WindVel', 'X_deg', 'Y_deg', 'id', 'Admin1', 'Cat_1', 'Cat_2', 'Cat_3', 'Unc', 'Prim', 'Sec', 'GridPenalty', 'WindCF', 'PopStartYear', 'ElecPopCalib', 'Pop2025', 'Pop2030', 'Pop2020', 'ElecStart', 'GridDistCalibElec', 'FinalElecCode2020', 'Commercial_Multiplier', 'MVConnectDist', 'NewConnections2025', 'NumPeoplePerHH', 'EnergyPerSettlement2025', 'TotalEnergyPerCell', 'Tier', 'AverageToPeak', 'SADieselFuelCost2025', 'MGDieselFuelCost2025', 'WindRenewableShare2025', 'WindHybridGenCapex2025', 'WindHybridEmissionFactor2025', 'WindHybridWindCapacity2025', 'WindHybridDieselCapacity2025', 'WindHybridBatteryCapacity202

In [7]:
# Check the unique values in the 'Admin1' column
print("Unique regions found in 'Admin1':")
print(onsseter.df['Admin1'].unique())

Unique regions found in 'Admin1':
<ArrowStringArray>
[              'Diana',                'Sava',               'Sofia',
        'Analanjirofo',               'Boeny',              'Melaky',
     'Alaotra Mangoro',           'Betsiboka',          'Atsinanana',
          'Analamanga',           'Bongolava',              'Menabe',
               'Itasy',      'Vakinankaratra',      'Amoron I Mania',
 'Vatovavy Fitovinany',     'Haute Matsiatra',    'Atsimo Andrefana',
            'Ihorombe',   'Atsimo Atsinanana',               'Anosy',
              'Androy']
Length: 22, dtype: str


In [8]:
# 1. Transfer the real names to the column the engine expects
onsseter.df['Admin_1'] = onsseter.df['Admin1']

# 2. Map the raw population count to the 'ElecPop' column 
# (This ensures the AI knows how many people it's talking about)
onsseter.df['ElecPop'] = onsseter.df['Pop']

# 3. Final Verification: Count the settlements in Sofia
sofia_check = onsseter.df[onsseter.df['Admin_1'] == 'Sofia'].shape[0]

print(f"--- DATA VERIFIED ---")
print(f"Success! Found {sofia_check} settlements in the Sofia region.")
print(f"Total population across all regions: {onsseter.df['ElecPop'].sum():,.0f}")

--- DATA VERIFIED ---
Success! Found 18387 settlements in the Sofia region.
Total population across all regions: 21,190,678


In [9]:
# Save the 'Conditioned' data to your data folder
output_path = 'data/Madagascar_Cleaned_Baseline.csv'
onsseter.df.to_csv(output_path, index=False)

print(f"--- SPRINT LOG: DAY 1 COMPLETE ---")
print(f"Cleaned data saved to: {output_path}")
print(f"Next step: Building the 'Brain' (Week 2)")

--- SPRINT LOG: DAY 1 COMPLETE ---
Cleaned data saved to: data/Madagascar_Cleaned_Baseline.csv
Next step: Building the 'Brain' (Week 2)


In [10]:
import pandas as pd

# 1. Load the cleaned baseline directly
# This pulls the 18,387 Sofia settlements into memory
file_path = 'data/Madagascar_Cleaned_Baseline.csv'
df = pd.read_csv(file_path)

# 2. Isolate Sofia for today's filtering
sofia_df = df[df['Admin_1'] == 'Sofia'].copy()

# 3. Health Check
print("--- COCKPIT STATUS: ACTIVE ---")
print(f"Total settlements loaded: {df.shape[0]}")
print(f"Settlements in Sofia: {sofia_df.shape[0]}")

print("\nCritical Column Check:")
cols_to_check = ['Admin_1', 'AgriDemand', 'ElecPop', 'CurrentMVLineDist', 'GHI']
for col in cols_to_check:
    status = "✅ Found" if col in sofia_df.columns else "❌ MISSING"
    print(f"{col}: {status}")

--- COCKPIT STATUS: ACTIVE ---
Total settlements loaded: 155275
Settlements in Sofia: 18387

Critical Column Check:
Admin_1: ✅ Found
AgriDemand: ✅ Found
ElecPop: ✅ Found
CurrentMVLineDist: ✅ Found
GHI: ✅ Found


In [11]:
# 1. Filter out the "zeros" - we only want places with actual demand and people
active_sites = sofia_df[(sofia_df['AgriDemand'] > 0) & (sofia_df['ElecPop'] > 0)].copy()

# 2. Re-calculate the Priority Score on only the active sites
# We'll also multiply by GHI (Solar) to ensure we favor the sunniest spots
active_sites['AgroLink_Priority'] = (active_sites['AgriDemand'] * active_sites['ElecPop'] * active_sites['CurrentMVLineDist'] * active_sites['GHI']) / 1000

# 3. Check how many potential hubs we actually have left
remaining_count = active_sites.shape[0]

print(f"--- FILTERING COMPLETE ---")
print(f"Out of 18,387 settlements, {remaining_count} have recorded Agricultural Demand.")

if remaining_count > 0:
    # 4. Find the Real Top 20
    real_top_20 = active_sites.sort_values(by='AgroLink_Priority', ascending=False).head(20)
    print("\n--- THE REAL SOFIA TOP 5 (Ranked by Need & Potential) ---")
    print(real_top_20[['id', 'AgriDemand', 'ElecPop', 'AgroLink_Priority']].head())
else:
    print("\n⚠️ WARNING: No settlements found with AgriDemand > 0.")
    print("We may need to check the column names or look at a different energy scenario.")

--- FILTERING COMPLETE ---
Out of 18,387 settlements, 0 have recorded Agricultural Demand.

⚠️ WARNING: No settlements found with AgriDemand > 0.
We may need to check the column names or look at a different energy scenario.


In [12]:
# 1. Let's see what the MAX value of AgriDemand is across the WHOLE country
max_agri = df['AgriDemand'].max()

# 2. Let's see if there are other columns that might contain our "Agro" data
# We'll search for any column names containing 'Agri', 'Irr', or 'Demand'
potential_cols = [c for c in df.columns if any(word in c for word in ['Agri', 'Irr', 'PUE', 'Demand'])]

print(f"--- DATA INVESTIGATION ---")
print(f"Maximum AgriDemand value in the entire file: {max_agri}")
print(f"\nOther potential columns found: {potential_cols}")

# 3. Look at a few random rows that actually have SOME data
# If AgriDemand is 0 everywhere, we'll check if 'PUE' (Productive Use of Energy) is used instead
print("\nColumn Statistics (First 10 columns):")
print(df.iloc[:, :10].describe().loc['max'])

--- DATA INVESTIGATION ---
Maximum AgriDemand value in the entire file: 0

Other potential columns found: ['AgriDemand', 'CommercialDemand', 'EducationDemand', 'HealthDemand', 'PerCapitaDemand', 'ResidentialDemandTierCustom', 'PerHouseholdDemand']

Column Statistics (First 10 columns):
AgriDemand                0.00000
CommercialDemand          0.00000
CurrentHVLineDist       682.54900
CurrentMVLineDist       593.84600
EducationDemand      123808.00000
GHI                    2271.43530
GridCellArea             62.36200
HealthDemand         257325.00000
HydropowerDist          100.79635
Name: max, dtype: float64


In [13]:
# 1. Create a "Social Anchor" score (Schools + Clinics)
# We add +1 to avoid any math issues with settlements that have zero schools
sofia_df['Social_Anchor'] = (sofia_df['EducationDemand'] + sofia_df['HealthDemand']) + 1

# 2. Calculate the AgroLink Proxy Score
# Formula: (People * Sun * Social Importance) / Distance to Grid
# We use (Dist + 1) to avoid dividing by zero if someone is ON the grid
sofia_df['AgroLink_Potential'] = (sofia_df['ElecPop'] * sofia_df['GHI'] * sofia_df['Social_Anchor']) / (sofia_df['CurrentMVLineDist'] + 1)

# 3. Normalize the score to a 0-100 scale (makes it easier for your "Chat" agent to explain)
max_score = sofia_df['AgroLink_Potential'].max()
sofia_df['AgroLink_Score'] = (sofia_df['AgroLink_Potential'] / max_score) * 100

# 4. Find the Real "Top 5"
top_5 = sofia_df.sort_values(by='AgroLink_Score', ascending=False).head(5)

print("--- GEOSPATIAL PROXY MODEL: COMPLETE ---")
print(f"Successfully calculated scores for {sofia_df.shape[0]} settlements.")
print("\n--- THE SOFIA 'AGRO-LINK' DIAMONDS ---")
print(top_5[['id', 'ElecPop', 'Social_Anchor', 'AgroLink_Score']])

--- GEOSPATIAL PROXY MODEL: COMPLETE ---
Successfully calculated scores for 18387 settlements.

--- THE SOFIA 'AGRO-LINK' DIAMONDS ---
            id    ElecPop  Social_Anchor  AgroLink_Score
125318  125389  25129.867        16973.5      100.000000
124905  124976  19851.309        17703.5       72.499204
125660  125732  13919.011        16061.0       71.028344
125094  125165  10682.646         5293.5       13.961055
125953  126025  13567.808         2373.5       10.311634


In [14]:
# 1. Get the Full Top 20
top_20_hubs = sofia_df.sort_values(by='AgroLink_Score', ascending=False).head(20).copy()

# 2. Add a Rank column for the final display
top_20_hubs['Rank'] = range(1, 21)

# 3. Calculate the "Impact Narrative"
total_pop = top_20_hubs['ElecPop'].sum()
avg_solar = top_20_hubs['GHI'].mean()

print("--- SOFIA AGRO-LINK MASTER LIST (TOP 20) ---")
print(top_20_hubs[['Rank', 'id', 'ElecPop', 'Social_Anchor', 'AgroLink_Score']])

print("\n--- IMPACT METRICS FOR PRESENTATION ---")
print(f"Total Population Served: {total_pop:,.0f} people")
print(f"Average Solar Irradiance: {avg_solar:.2f} kWh/m²")
print(f"Target Region: Sofia, Madagascar")

--- SOFIA AGRO-LINK MASTER LIST (TOP 20) ---
        Rank      id     ElecPop  Social_Anchor  AgroLink_Score
125318     1  125389  25129.8670        16973.5      100.000000
124905     2  124976  19851.3090        17703.5       72.499204
125660     3  125732  13919.0110        16061.0       71.028344
125094     4  125165  10682.6460         5293.5       13.961055
125953     5  126025  13567.8080         2373.5       10.311634
124711     6  124782   3054.4350        16061.0        9.007997
143362     7  143437   2998.7546         3651.0        2.902994
125548     8  125619   6958.6826         1461.0        2.758274
124602     9  124673   7296.9770         1461.0        2.044854
125187    10  125258   2796.9830         3103.5        1.926067
17367     11   17387   1455.3363         6571.0        1.901889
143687    12  143762   2110.5435         2921.0        1.632223
125216    13  125287   4329.2850         1461.0        1.543757
125588    14  125659   3303.6702         1461.0        1.38

In [15]:
# 1. Print the full list of columns
print("--- FULL VARIABLE LIST ---")
all_cols = df.columns.tolist()
for i, col in enumerate(all_cols):
    print(f"{i+1}. {col}")

# 2. Get the technical summary (Data types and Non-Null counts)
print("\n--- TECHNICAL DATA SUMMARY ---")
print(df.info())

# 3. See the "Macro Stats" for every numeric column
# This shows you the min, max, and average for everything at once
full_summary = df.describe()
print("\n--- MACRO STATISTICS PREVIEW ---")
print(full_summary)

--- FULL VARIABLE LIST ---
1. AgriDemand
2. CommercialDemand
3. Country
4. CurrentHVLineDist
5. CurrentMVLineDist
6. EducationDemand
7. GHI
8. GridCellArea
9. HealthDemand
10. HydropowerDist
11. IsUrban
12. NightLights
13. PerCapitaDemand
14. PlannedHVLineDist
15. PlannedMVLineDist
16. Pop
17. ResidentialDemandTierCustom
18. RoadDist
19. SubstationDist
20. TransformerDist
21. TravelHours
22. WindVel
23. X_deg
24. Y_deg
25. id
26. Admin1
27. Cat_1
28. Cat_2
29. Cat_3
30. Unc
31. Prim
32. Sec
33. GridPenalty
34. WindCF
35. PopStartYear
36. ElecPopCalib
37. Pop2025
38. Pop2030
39. Pop2020
40. ElecStart
41. GridDistCalibElec
42. FinalElecCode2020
43. Commercial_Multiplier
44. MVConnectDist
45. NewConnections2025
46. NumPeoplePerHH
47. EnergyPerSettlement2025
48. TotalEnergyPerCell
49. Tier
50. AverageToPeak
51. SADieselFuelCost2025
52. MGDieselFuelCost2025
53. WindRenewableShare2025
54. WindHybridGenCapex2025
55. WindHybridEmissionFactor2025
56. WindHybridWindCapacity2025
57. WindHybridDie

In [16]:
def chat_onsset_v2(user_query):
    query = user_query.lower()
    
    print(f"--- 🤖 AGENT REASONING: '{user_query}' ---")
    
    # INTENT 1: The "Best Overall" (Your custom Agro-Link Score)
    if any(word in query for word in ["best", "priority", "top", "rank"]):
        top_site = top_20_hubs.iloc[0]
        return (f"Based on my Geospatial Proxy Model, the #1 priority for Sofia is Settlement {top_site['id']}. "
                f"It has an AgroLink Score of {top_site['AgroLink_Score']:.1f}/100 and serves {top_site['ElecPop']:,.0f} people.")

    # INTENT 2: The "Impact" (Pure Population)
    elif any(word in query for word in ["impact", "people", "population", "size"]):
        most_populous = sofia_df.sort_values(by='ElecPop', ascending=False).iloc[0]
        return (f"If maximizing human impact is the goal, Settlement {most_populous['id']} is the leader "
                f"with a population of {most_populous['ElecPop']:,.0f}. However, check its grid distance before investing.")

    # INTENT 3: The "Off-Grid" (Distance from Infrastructure)
    elif any(word in query for word in ["isolated", "off-grid", "far", "remote"]):
        most_remote = sofia_df.sort_values(by='CurrentMVLineDist', ascending=False).iloc[0]
        return (f"The most remote settlement is {most_remote['id']}, located {most_remote['CurrentMVLineDist']:.1f}km from the nearest grid line. "
                f"This is a prime candidate for a fully autonomous solar mini-grid.")

    # INTENT 4: The "Solar Resource" (GHI)
    elif any(word in query for word in ["sun", "solar", "energy", "resource"]):
        best_sun = sofia_df.sort_values(by='GHI', ascending=False).iloc[0]
        return (f"For maximum solar efficiency, Settlement {best_sun['id']} has the highest irradiance at {best_sun['GHI']:.2f} kWh/m². "
                f"Your solar pumps will perform at peak capacity here.")

    # FALLBACK
    else:
        return ("I'm not quite sure how to answer that. Try asking about 'top sites', 'remote areas', 'population impact', or 'solar potential'.")

# --- TEST DRIVE THE BRAIN ---
print(chat_onsset_v2("Which villages are most isolated?"))
print("\n" + chat_onsset_v2("Where can we have the most impact?"))

--- 🤖 AGENT REASONING: 'Which villages are most isolated?' ---
The most remote settlement is 9336, located 401.2km from the nearest grid line. This is a prime candidate for a fully autonomous solar mini-grid.
--- 🤖 AGENT REASONING: 'Where can we have the most impact?' ---

If maximizing human impact is the goal, Settlement 125389 is the leader with a population of 25,130. However, check its grid distance before investing.


In [17]:
def get_site_card_final(site_id):
    site_data = sofia_df[sofia_df['id'] == int(site_id)].iloc[0]
    
    # Use the PV Hybrid Capex we found in the autopsy
    pv_capex = site_data['PVHybridGenCapex2025']
    lcoe = site_data['MinimumOverallLCOE2025']

    print(f"--- 💎 BANKABILITY CARD: SETTLEMENT {site_id} ---")
    print(f"{'Location:':<22} {site_data['Admin_1']} Region")
    print(f"{'AgroLink Score:':<22} {site_data['AgroLink_Score']:.2f} / 100")
    print("-" * 45)
    
    print(f"📈 IMPACT & GROWTH")
    print(f"{'Current Pop:':<22} {site_data['ElecPop']:,.0f} people")
    print(f"{'2030 Projection:':<22} {site_data['Pop2030']:,.0f} people")
    print(f"{'Pop Growth Factor:':<22} +{((site_data['Pop2030']/site_data['ElecPop'])-1)*100:.1f}%")
    
    print(f"\n☀️ TECH & LOGISTICS")
    print(f"{'System Type:':<22} PV-Diesel Hybrid Mini-Grid")
    print(f"{'Solar Potential:':<22} {site_data['GHI']:.2f} kWh/m²")
    print(f"{'Grid Isolation:':<22} {site_data['CurrentMVLineDist']:.1f} km from grid")
    
    print(f"\n💰 FINANCING METRICS")
    print(f"{'Est. System Capex:':<22} ${pv_capex:,.2f}")
    print(f"{'LCOE (Unit Cost):':<22} ${lcoe:.3f} per kWh")
    print("-" * 45)
    print("STRATEGIC VERDICT: High-capacity hub with elite growth.")
    print("Ideal for anchor-tenant financing (Schools + Ag-Pumps).")
    
    return "✅ Final Bankability Card generated."

print(get_site_card_final(125389))

--- 💎 BANKABILITY CARD: SETTLEMENT 125389 ---
Location:              Sofia Region
AgroLink Score:        100.00 / 100
---------------------------------------------
📈 IMPACT & GROWTH
Current Pop:           25,130 people
2030 Projection:       49,557 people
Pop Growth Factor:     +97.2%

☀️ TECH & LOGISTICS
System Type:           PV-Diesel Hybrid Mini-Grid
Solar Potential:       2229.67 kWh/m²
Grid Isolation:        270.5 km from grid

💰 FINANCING METRICS
Est. System Capex:     $13,143,229.00
LCOE (Unit Cost):      $0.144 per kWh
---------------------------------------------
STRATEGIC VERDICT: High-capacity hub with elite growth.
Ideal for anchor-tenant financing (Schools + Ag-Pumps).
✅ Final Bankability Card generated.


In [18]:
import os

# 1. Create the 'outputs' folder if it doesn't exist
if not os.path.exists('outputs'):
    os.makedirs('outputs')
    print("📁 Created 'outputs' folder.")

# 2. Re-exporting with the new financial insights
with open('outputs/Project_Impact_Summary.txt', 'w') as f:
    f.write("--- CHATONSSET PROJECT IMPACT SUMMARY (SOFIA REGION) ---\n")
    f.write(f"Region: Sofia, Madagascar\n")
    f.write(f"Total Population Impact (Top 20): {total_pop:,.0f} people\n")
    
    # Getting the specific hub data
    hub_id = 125389
    hub_capex = sofia_df[sofia_df['id'] == hub_id]['PVHybridGenCapex2025'].values[0]
    
    f.write(f"Primary Hub (ID {hub_id}) PV Hybrid Capex: ${hub_capex:,.2f}\n")
    f.write(f"Average Solar Irradiance: {avg_solar:.2f} kWh/m2\n")
    f.write("--- END OF SUMMARY ---")

print("🚀 Sunday Assets Frozen and Exported. You are ready for the May 21st internal review.")

🚀 Sunday Assets Frozen and Exported. You are ready for the May 21st internal review.


In [19]:
# Based on your previous code, your DataFrame is 'sofia_df'
sofia_df.to_csv('madagascar_sofia_calibrated.csv', index=False)

print("✅ SUCCESS: Data Engine Exported.")
print("You can now close this notebook and head to '3. ChatOnSSET_Agent'.")

✅ SUCCESS: Data Engine Exported.
You can now close this notebook and head to '3. ChatOnSSET_Agent'.


# 2. Start year

Select the start year of the analysis, for which you have baseline (current) demographic and electrification data.

In [ ]:
start_year = 2024

# 3. Enter baseline country specific data

Enter the demographic and grid electrification status parameters

### Demographics and Social components - Current

In [ ]:
pop_start_year = 20800000       ### Write the population in the base year (e.g. 2024) 

urban_ratio_start_year = 0.34 ### Write the urban population population ratio in the base year (e.g. 2024)

grid_elec_ratio_start_year = 0.539   ### Write the grid electrification rate in the base year (e.g. 2024)
grid_urban_elec_ratio = 0.80      ### Write urban grid electrification rate in the base year (e.g. 2024)
grid_rural_elec_ratio = 0.34         ### Write rural grid electrification rate in the base year (e.g. 2024)

# 4. Calibration of start year values and general information

The following steps calibrate the start year conditions in the country in terms of population and current electrification rate, and also adds some additional useful information to be used in the further calculations. If you have retrieved the input file from Energydata.info some of these steps are already completed, and you may skip those cells below where there is a note. 

In [ ]:
pop_modelled, urban_modelled = onsseter.calibrate_current_pop_and_urban(pop_start_year, urban_ratio_start_year)
onsseter.df[SET_POP + "{}".format(start_year)] = onsseter.df['PopStartYear'] 
onsseter.df[SET_WINDCF] = onsseter.calc_wind_cfs(onsseter.df[SET_WINDVEL])

display(Markdown('#### The csv file has been imported correctly. Here is a preview:'))
display(onsseter.df[['Country','Pop', 'GridCellArea', 'NightLights', 'TravelHours', 'GHI', 'WindVel', 'CurrentMVLineDist']].sample(7))

#### Calibration of currently grid-electrified settlements

The model calibrates which settlements are likely to be electrified in the start year, to match the national statistical values defined above. A settlement is considered to be electrified if it meets all of the following conditions:
- Has more night-time lights than the defined threshold (this is set to 0 by default. Set to -1 to disregard NTL completely)
- Is closer to the existing grid network than the distance limit
- Has more population than the threshold

First, define the threshold limits. Then run the calibration and check if the results seem okay. Else, redefine these thresholds and run again.

In [ ]:
min_night_lights = 0    ### 0 Indicates no night light, while any number above refers to the night-lights intensity
min_pop = 50      ### Settlement population above which we can assume that it could be electrified
max_mv_line_distance = 2 ### Distance in km from the existing grid network below which we can assume a settlement could be electrified
max_transformer_dist = 1

elec_calibration_results = onsseter.calibrate_grid_elec_current(grid_elec_ratio_start_year, grid_urban_elec_ratio, grid_rural_elec_ratio, start_year, 
                                                                min_night_lights=min_night_lights, min_pop=min_pop, max_transformer_dist=max_transformer_dist, 
                                                                max_mv_dist=max_mv_line_distance, max_hv_dist=5, buffer=False)

In [ ]:
mg_dist = 0.5   # Distance from existing mini-grids to consider settlements connected to the mini-grid
mg_ntl = 0   # Night-time light threshold to consider a settlement mini-grid electrified, in combination with mg_dist. 0 means NTL is not required, any higher value means a higher cut-off for settlements with NTL within dist is electrified
mg_pop = 0  # Settlement population above which we can assume that it could be electrified by mini-grid 

onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)] == 5, 'ElecStart'] = 0
onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)] == 5, 'FinalElecCode' + '{}'.format(start_year)] = 99
onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)] == 5, 'ElecPopCalib'] = 0
onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)] == 5, 'ElecPop' + '{}'.format(start_year)] = 0

onsseter.df.loc[(onsseter.df['FinalElecCode' + '{}'.format(start_year)] != 1) & (onsseter.df['NightLights'] >= mg_ntl) & (onsseter.df['FinalElecCode' + '{}'.format(start_year)] != 1) &
                (onsseter.df['MGDist'] < mg_dist) & (onsseter.df[SET_POP + "{}".format(start_year)] > mg_pop), 'ElecStart'] = 1
onsseter.df.loc[(onsseter.df['FinalElecCode' + '{}'.format(start_year)] != 1) & (onsseter.df['NightLights'] >= mg_ntl) & (onsseter.df['FinalElecCode' + '{}'.format(start_year)] != 1) &
                (onsseter.df['MGDist'] < mg_dist) & (onsseter.df[SET_POP + "{}".format(start_year)] > mg_pop), 'FinalElecCode' + '{}'.format(start_year)] = 5
onsseter.df.loc[(onsseter.df['FinalElecCode' + '{}'.format(start_year)] != 1) & (onsseter.df['NightLights'] >= mg_ntl) & (onsseter.df['FinalElecCode' + '{}'.format(start_year)] != 1) &
                (onsseter.df['MGDist'] < mg_dist) & (onsseter.df[SET_POP + "{}".format(start_year)] > mg_pop), 'ElecPopCalib'] = onsseter.df['PopStartYear']
onsseter.df['ElecPop' + '{}'.format(start_year)] = onsseter.df['ElecPopCalib']

mg_pop = onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)] == 5, 'PopStartYear'].sum() / onsseter.df['PopStartYear'].sum()

pop_sum = onsseter.df['PopStartYear'].sum()
elec_pop_sum = onsseter.df[SET_ELEC_POP_CALIB].sum()

print('The modelled grid + mini-grid electrification rate is: ', round(elec_pop_sum/pop_sum, 3))

#### Visualization

The figure below show the results of the calibration. Settlements in **blue** are considered to be (at least partly) grid-electrified already in the start year of the analysis, settlements in **purple** are considered to be connected to mini-grids at the start of the analysis, while settlements in **grey** are yet to be electrified. Re-running the calibration step with different intial values may change the map below.

In [ ]:
colors = ['#D3D3D3','#808080', 'red']
plt.figure(figsize=(12,12))
plt.plot(onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)]==99, SET_X_DEG], onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)]==99, SET_Y_DEG], color='#D3D3D3', marker=',', linestyle='none')
plt.plot(onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)]==1, SET_X_DEG], onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)]==1, SET_Y_DEG], color='#00008B', marker='o', ms=1, linestyle='none')
plt.plot(onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)]==5, SET_X_DEG], onsseter.df.loc[onsseter.df['FinalElecCode' + '{}'.format(start_year)]==5, SET_Y_DEG], color='red', marker='o', ms=2, linestyle='none')
if onsseter.df[SET_X_DEG].max() - onsseter.df[SET_X_DEG].min() > onsseter.df[SET_Y_DEG].max() - onsseter.df[SET_Y_DEG].min():
    plt.xlim(onsseter.df[SET_X_DEG].min() - 1, onsseter.df[SET_X_DEG].max() + 1)
    plt.ylim((onsseter.df[SET_Y_DEG].min()+onsseter.df[SET_Y_DEG].max())/2 - 0.5*abs(onsseter.df[SET_X_DEG].max() - onsseter.df[SET_X_DEG].min()) - 1, (onsseter.df[SET_Y_DEG].min()+onsseter.df[SET_Y_DEG].max())/2 + 0.5*abs(onsseter.df[SET_X_DEG].max() - onsseter.df[SET_X_DEG].min()) + 1)
else:
    plt.xlim((onsseter.df[SET_X_DEG].min()+onsseter.df[SET_X_DEG].max())/2 - 0.5*abs(onsseter.df[SET_Y_DEG].max() - onsseter.df[SET_Y_DEG].min()) - 1, (onsseter.df[SET_X_DEG].min()+onsseter.df[SET_X_DEG].max())/2 + 0.5*abs(onsseter.df[SET_Y_DEG].max() - onsseter.df[SET_Y_DEG].min()) + 1)
    plt.ylim(onsseter.df[SET_Y_DEG].min() -1, onsseter.df[SET_Y_DEG].max() +1)
plt.figure(figsize=(30,30))
logging.getLogger('matplotlib.font_manager').disabled = True
plt.show()

onsseter.df['Technology{}'.format(start_year)] = 'Unelectrified'
onsseter.df.loc[onsseter.df[SET_ELEC_FINAL_CODE + "{}".format(start_year)] == 1, 'Technology{}'.format(start_year)] = 'Existing grid'
onsseter.df.loc[onsseter.df[SET_ELEC_FINAL_CODE + "{}".format(start_year)] == 2, 'Technology{}'.format(start_year)] = 'Grid extension'
onsseter.df.loc[onsseter.df[SET_ELEC_FINAL_CODE + "{}".format(start_year)] == 3, 'Technology{}'.format(start_year)] = 'SHS'
onsseter.df.loc[onsseter.df[SET_ELEC_FINAL_CODE + "{}".format(start_year)] == 5, 'Technology{}'.format(start_year)] = 'PV Hybrid Mini-Grid'
onsseter.df.loc[onsseter.df[SET_ELEC_FINAL_CODE + "{}".format(start_year)] == 6, 'Technology{}'.format(start_year)] = 'Wind Hybrid Mini-Grid'
onsseter.df.loc[onsseter.df[SET_ELEC_FINAL_CODE + "{}".format(start_year)] == 7, 'Technology{}'.format(start_year)] = 'Hydro Mini-Grid'

## 5. Exporting calibrated file

This code saves the calibrated file, which will be used to run scenarios in the next step.

**Note that if you do not change the scenario name, the previous output files will be overwritten**

In [ ]:
file_name = "OnSSET_InputFile_Calibrated"

In [ ]:
messagebox.showinfo('OnSSET', 'Browse to the folder where you want to save the outputs')

output_dir = filedialog.askdirectory()
output_dir_results = os.path.join(output_dir, '{}.csv'.format(file_name))
onsseter.df.to_csv(output_dir_results, float_format="%.4f", index=False)